# 3. Secure aggregation and platform proof
Run this notebook on the LinuxONE VM and keep the output as evidence.

In [ ]:
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make core/ importable from notebooks/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.facecolor": "#0A1830", "axes.facecolor": "#12264A", "axes.edgecolor": "#2A4170",
    "text.color": "#E8EFFA", "axes.labelcolor": "#E8EFFA", "xtick.color": "#8FA3C4", "ytick.color": "#8FA3C4",
    "axes.prop_cycle": plt.cycler(color=["#34D3C0", "#6EA8FF", "#FFB547", "#B58CFF"]), "axes.grid": True, "grid.color": "#2A4170"})
from core.data_gen import load_all, FEATURES
banks, test = load_all()

In [ ]:
from core.federated import run_federation
res = run_federation(banks, test, epsilon=5, rounds=5); s = res['sample']
print('Masks cancel, merged model is exact:', np.allclose(s['agg'], s['exact']))
pd.DataFrame({f'bank{i} true': s['true'][i][:4] for i in range(3)} | {f'bank{i} masked': s['masked'][i][:4] for i in range(3)})

## What machine is this?

In [ ]:
from core.platform_proof import platform_report, scoring_benchmark
rep = platform_report(); {k: v for k, v in rep.items() if k != 'cpu_features'}

`secure_execution_guest` is `'1'` only inside a Secure Execution guest. If it is `None`, do not claim Secure Execution.

In [ ]:
b = pd.DataFrame(scoring_benchmark(res['weights'], 0.0, len(FEATURES))).round(3); b

In [ ]:
plt.figure(figsize=(7, 3.8)); plt.bar(b['batch'].astype(str), b['txns_per_sec'])
plt.yscale('log'); plt.xlabel('batch size'); plt.ylabel('transactions / second'); plt.show()